In [61]:
import pytesseract
from PIL import Image

print(f"Tesseract Version: {pytesseract.get_tesseract_version()}")

Tesseract Version: 5.5.2


In [62]:
img = Image.open('../../data/1/R_9346_I_1_0001.jpg')

# OEM => OCR Engine Mode (0-3) 3 erkennt automatisch was es brauch
# PSM => Page Segmentation Mode (1-13)
    # 1 = vollautomatische seitenaalyse mit osd
    # 3 = vollautomatische seitenaalyse ohne osd (standardwert)
    # 4 = einzelne TExtplatze mit variablen Schriftgrößen
    # 6 = einzelnen, textblock
    # 11 = findet so viel text wie möglich (sparse text) 
custom_config = r'--oem 3 psm 6'


text = pytesseract.image_to_string(img, lang='deu_frak', config=custom_config) # oder 'deu'
print(text)


# Pandas Dataframe mit Bounding Boxes und COnfidence Scire
test = pytesseract.image_to_data(img, lang='deu', config=custom_config)
print (test)


.Tf-Ur.:

Zulassunggkarten für Bildstreifen find öffentl. Urkunden im Sinne de-

Aenderungen durfen nur von der FtlmsPrufstelle vorgenommen werden. J

« Ursprungs-Firma: »

§ 267 Reichs- Strafgesetzbuchs Ohne amtlichem Stempel sind sie ungültig. s

soispar Film S. m. b. ii., Berlin sW 48

Priedrichstrasse 25

Die Wölfin.

. Titel des Bildes:
L « Sensationsschauspiel in 5 Akten von Jos. Delmont.
Hergestellt von der Solar-Film G. m. b. H» Berlin.
Spielleitung: Rolf Brunnen
Photographie: August Brückner.
Architektur: Alfred Columbu5.
Untertitel. cHist-inetwerzeichniz:
Lady Florence. . . ........ ". .. Ressel era
Lord William Heinrich Peer
Lord Henry ................ Fred Selva-Goebels
(Tri’anon-Theater)
Thurfton, Diener ............ Walter Formes
» Dr. Munfon, Jsrrenarzt ........ Heinz Burkart

(Deutsches Theater)
Jack ...................... Paul Passarge
Die Wölfin: Lady Florence — Ressel Orla.


level	page_num	block_num	par_num	line_num	word_num	left	top	width	height	conf	text
1	1	0	0	0

In [63]:
#pdf_bytes = pytesseract.image_to_pdf_or_hocr(img, lang='deu', config=custom_config, extension='pdf')

# 2. Daten als physische Datei speichern
#output_path = 'ausgabe_dokument.pdf'
#with open(output_path, 'wb') as f:
#    f.write(pdf_bytes)

#print(f"PDF erfolgreich unter '{output_path}' gespeichert.")

# Fiftyone Dataset

In [42]:
import fiftyone as fo
import pytesseract
from PIL import Image

dataset = fo.Dataset.from_dir(
    dataset_dir="../../data/1",
    dataset_type=fo.types.ImageDirectory,
    name="OCR_Test"
)

for sample in dataset:
    filepath = sample.filepath
    img = Image.open(filepath)
    img_width, img_height = img.size
    
    # Nutze image_to_data, um Bounding Boxes und Konfidenzwerte zu erhalten
    ocr_data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)
    
    detections = []
    n_boxes = len(ocr_data['text'])
    
    for i in range(n_boxes):
        text = ocr_data['text'][i].strip()
        conf = int(ocr_data['conf'][i])
        
        # Filtere leere Erkennungen und Artefakte (Konfidenz < 0 bedeutet oft kein Text)
        if not text or conf < 0:
            continue
            
        # Absolute Tesseract-Koordinaten in Pixeln
        x = ocr_data['left'][i]
        y = ocr_data['top'][i]
        w = ocr_data['width'][i]
        h = ocr_data['height'][i]
        
        # Relative FiftyOne-Koordinaten (0.0 bis 1.0)
        rel_x = x / img_width
        rel_y = y / img_height
        rel_w = w / img_width
        rel_h = h / img_height
        
        # Erstelle die Detection
        detection = fo.Detection(
            label=text,
            bounding_box=[rel_x, rel_y, rel_w, rel_h],
            confidence=conf / 100.0  # FiftyOne erwartet oft Konfidenz zwischen 0 und 1
        )
        detections.append(detection)
        
    # Speichere die Liste der Detections im Sample unter einem neuen Feldnamen
    sample["tesseract_ocr"] = fo.Detections(detections=detections)
    sample.save()

print("OCR-Daten erfolgreich in das Dataset geladen.")

# 3. Starte die FiftyOne App zur Visualisierung
session = fo.launch_app(dataset)
session.wait()

FiftyOneConfigError: MongoDB could not be installed on your system. Please define a `database_uri` in your `fiftyone.core.config.FiftyOneConfig` to connect to yourown MongoDB instance or cluster 